### **Panhuman library append Ion Mobility**

**Note:** This is run on the boltzmann cluster on the GPU so it is quite fast.

In [1]:
import pandas as pd
from peptdeep.pretrained_models import ModelManager
import re

/banana/rostlab/jcharkow/.conda/envs/peptdeep/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/banana/rostlab/jcharkow/.conda/envs/peptdeep/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### **Functions**

In [2]:
def modified_sequence_to_tuple(seq):
    return modified_sequence_to_tuple_helper(seq, '', '')

def modified_sequence_to_tuple_helper(seq, mods, pos, translate_mod_dict = {'(UniMod:4)':'Carbamidomethyl@C', '(UniMod:35)':'Oxidation@M'}):
    searchRslt =  re.search(r'\(UniMod:\d{1,2}\)', seq)
    if searchRslt == None:
        return (seq, mods, pos)
    
    else:
        unimod = seq[searchRslt.span()[0]:searchRslt.span()[1]]
        if mods != '':
            mods += ';'
            pos += ';'
        mods += translate_mod_dict[unimod]
        pos += str(searchRslt.span()[0])
        seq = seq[:searchRslt.span()[0]] + seq[searchRslt.span()[1]:]
        return modified_sequence_to_tuple_helper(seq, mods, pos, translate_mod_dict)

#### **Load Data**

In [5]:
lib = pd.read_csv("../../data/PanHuman/phl004_s32.csv", sep='\t') 

#### **Only 2 types of Modifications are present**

In [5]:
lib['FullUniModPeptideName'].str.extract(r'(\(UniMod:\d*)\)').drop_duplicates()

,0
0,NaN
84,(UniMod:4
1008,(UniMod:35


This corresponds to carboxyinlyate and methylation, have to take these into account when predicting IM. 

---

#### **Create Model**

In [6]:
model_mgr = ModelManager(device='gpu')
model_mgr.nce = 30
model_mgr.instrument= 'timsTOF'
model_mgr.load_installed_models()

/banana/rostlab/jcharkow/.conda/envs/peptdeep/lib/python3.9/site-packages/peptdeep/model/model_interface.py:731: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(str

In [7]:
precursor_df = lib[['FullUniModPeptideName', 'PrecursorCharge']].rename(columns={'FullUniModPeptideName':'modified_sequence', 'PrecursorCharge':'charge'})
modification_tuple = precursor_df['modified_sequence'].apply(modified_sequence_to_tuple)
#precursor_df['mods'] = ''
#precursor_df['mod_sites'] = ''
precursor_df = precursor_df.drop_duplicates()

precursor_df['sequence'] = modification_tuple.apply(lambda x: x[0])
precursor_df['mods'] = modification_tuple.apply(lambda x: x[1])
precursor_df['mod_sites'] = modification_tuple.apply(lambda x: x[2])

In [8]:
model_mgr.predict_mobility(precursor_df)

2025-05-21 15:13:05> Predicting mobility ...


100%|███████████████████████████████████████████| 56/56 [00:11<00:00,  4.78it/s]


,modified_sequence,charge,sequence,mods,mod_sites,nAA,ccs_pred,precursor_mz,mobility_pred
0,RGLLQLL,2,RGLLQLL,,,7,313.639343,406.771263,0.769982
1,DC(UniMod:4)EVIER,2,DCEVIER,Carbamidomethyl@C,2,7,322.378906,460.710742,0.792983
2,DC(UniMod:4)FGC(UniMod:4)LR,2,DCFGCLR,Carbamidomethyl@C;Carbamidomethyl@C,2;5,7,317.623566,464.194205,0.781373
3,RYQEIDL,2,RYQEIDL,,,7,329.385010,468.742899,0.810422
4,KNIDSIL,2,KNIDSIL,,,7,299.529572,401.737086,0.735189
...,...,...,...,...,...,...,...,...,...
409119,KTEDSAPPKEPAVAAEQPKAEQPDGGAAGEEGAAAEEGPAAAQEGS...,6,KTEDSAPPKEPAVAAEQPKAEQPDGGAAGEEGAAAEEGPAAAQEGS...,,,63,1183.646606,955.274139,0.982744
409120,DEAAGGAAAAAAEAGAASGEQAAAPGEEAAAGEEGAAGGDPQEAKP...,6,DEAAGGAAAAAAEAGAASGEQAAAPGEEAAAGEEGAAGGDPQEAKP...,,,63,1235.097046,955.274139,1.025462
409121,RNPYYSNYQHNPFGHDHSNQQPGAYPELGGGGGSQLAALGPSQQGG...,7,RNPYYSNYQHNPFGHDHSNQQPGAYPELGGGGGSQLAALGPSQQGG...,,,63,1224.522339,922.862961,0.871681
409122,RSGSGGGYGGGSGGGSGSGGGYNGGSGGSHGGGSGGSHGGGSGGGY...,5,RSGSGGGYGGGSGGGSGSGGGYNGGSGGSHGGGSGGSHGGGSGGGY...,,,66,942.366333,995.393098,0.938554


In [9]:
lib.columns

Index(['PrecursorMz', 'ProductMz', 'Tr_recalibrated', 'transition_name', 'CE',
       'LibraryIntensity', 'transition_group_id', 'decoy', 'PeptideSequence',
       'ProteinName', 'Annotation', 'FullUniModPeptideName', 'MissedCleavages',
       'Replicates', 'NrModifications', 'PrecursorCharge', 'GroupLabel',
       'UniprotID'],
      dtype='object')

In [10]:
libOut = (precursor_df[['modified_sequence', 'charge', 'mobility_pred']].drop_duplicates().merge(lib, left_on=['modified_sequence', 'charge'], right_on=['FullUniModPeptideName', 'PrecursorCharge'])
          .rename(columns={'mobility_pred':'PrecursorIonMobility'}))

In [11]:
len(libOut)

2454768

In [12]:
len(lib)

2454768

Library in and out are the same length which is comforting. 

In [13]:
libOut.to_csv("phl004_s32_imAppended.csv", sep='\t', index=False)